# AuraLock LoRA Benchmark on Google Colab

Notebook nay chay benchmark DreamBooth/LoRA tren Colab free GPU theo flow cua AuraLock.

## Muc tieu
- mount Google Drive de doc artwork, checkpoint, va luu report
- cai AuraLock benchmark runtime tren Colab
- clone `Anti-DreamBooth` de lay script train/infer tham chieu
- chay `dry-run` truoc de khoa path va preflight
- chi chay `--execute` khi da san sang GPU va checkpoint


## Truoc khi chay

1. Mo `Runtime > Change runtime type > GPU`
2. Chuan bi mot thu muc Google Drive chua artwork dau vao
3. Chuan bi checkpoint Diffusers co `model_index.json` trong Drive, hoac dung cell download ben duoi
4. Chay tung cell theo thu tu, luon chay `dry-run` truoc `--execute`


In [ ]:
!nvidia-smi

import os
import subprocess
import sys
from pathlib import Path

print('Python:', sys.version)
print('CUDA_VISIBLE_DEVICES:', os.environ.get('CUDA_VISIBLE_DEVICES'))

In [ ]:
REPO_URL = "https://github.com/locfaker/AuraLock.git"
WORKSPACE = Path("/content/AuraLock")

if not WORKSPACE.exists():
    subprocess.run(["git", "clone", REPO_URL, str(WORKSPACE)], check=True)

%cd /content/AuraLock
!python -m pip install --upgrade pip
!pip install -e ".[benchmark]" datasets ftfy tensorboard


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
ANTI_DREAMBOOTH_ROOT = Path('/content/Anti-DreamBooth')

if not ANTI_DREAMBOOTH_ROOT.exists():
    !git clone https://github.com/VinAIResearch/Anti-DreamBooth.git /content/Anti-DreamBooth

print('Anti-DreamBooth root:', ANTI_DREAMBOOTH_ROOT)


In [ ]:
AURALOCK_ROOT = Path('/content/AuraLock')
ARTWORK_DIR = Path('/content/drive/MyDrive/auralock/artworks')
BENCHMARK_ROOT = Path('/content/drive/MyDrive/auralock/benchmark_runs/colab')
REPORT_PATH = BENCHMARK_ROOT / 'reports' / 'lora-manifest-colab.json'
MODEL_DIR = Path('/content/drive/MyDrive/auralock/models/sd15')
MODEL_ID = 'runwayml/stable-diffusion-v1-5'
INSTANCE_PROMPT = 'a sks painting'
CLASS_PROMPT = 'a painting'
PROFILE_LIST = 'safe,balanced,strong'

BENCHMARK_ROOT.mkdir(parents=True, exist_ok=True)
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

print('Artwork dir exists:', ARTWORK_DIR.exists())
print('Model dir exists:', MODEL_DIR.exists())
print('Report path:', REPORT_PATH)


In [ ]:
# Optional: download a Diffusers checkpoint into Google Drive if MODEL_DIR is empty.
# This can take a long time and requires enough Drive space.

from huggingface_hub import snapshot_download

if not (MODEL_DIR / 'model_index.json').exists():
    snapshot_download(
        repo_id=MODEL_ID,
        local_dir=str(MODEL_DIR),
        local_dir_use_symlinks=False,
        resume_download=True,
    )

print('Checkpoint ready:', (MODEL_DIR / 'model_index.json').exists())


## Dry-run truoc

Cell nay kiem tra path, preflight, va tao manifest benchmark ma chua train that. Day la buoc bat buoc truoc `--execute`.


In [ ]:
dry_run_command = f'''auralock benchmark-lora "{ARTWORK_DIR}" --work-dir "{BENCHMARK_ROOT}" --profiles "{PROFILE_LIST}" --pretrained-model-path "{MODEL_DIR}" --script-path "{ANTI_DREAMBOOTH_ROOT / 'train_dreambooth_lora.py'}" --infer-script-path "{ANTI_DREAMBOOTH_ROOT / 'infer.py'}" --instance-prompt "{INSTANCE_PROMPT}" --class-prompt "{CLASS_PROMPT}" --report "{REPORT_PATH}"'''

print(dry_run_command)
subprocess.run(dry_run_command, shell=True, check=True, cwd=AURALOCK_ROOT)


In [ ]:
import json

with REPORT_PATH.open('r', encoding='utf-8') as handle:
    payload = json.load(handle)

print('Ready:', payload['preflight']['ready'])
print('Missing modules:', payload['preflight']['missing_modules'])
print('Missing paths:', payload['preflight']['missing_paths'])
print('Invalid paths:', payload['preflight']['invalid_paths'])
print('Jobs:', len(payload['jobs']))


## Execute that su

Chi chay cell nay khi `dry-run` da sach va runtime Colab dang co GPU.


In [ ]:
RUN_EXECUTE = False

execute_command = f'''auralock benchmark-lora "{ARTWORK_DIR}" --execute --work-dir "{BENCHMARK_ROOT}" --profiles "{PROFILE_LIST}" --pretrained-model-path "{MODEL_DIR}" --script-path "{ANTI_DREAMBOOTH_ROOT / 'train_dreambooth_lora.py'}" --infer-script-path "{ANTI_DREAMBOOTH_ROOT / 'infer.py'}" --instance-prompt "{INSTANCE_PROMPT}" --class-prompt "{CLASS_PROMPT}" --report "{BENCHMARK_ROOT / 'reports' / 'lora-execute-colab.json'}"'''

print(execute_command)
if RUN_EXECUTE:
    subprocess.run(execute_command, shell=True, check=True, cwd=AURALOCK_ROOT)
else:
    print('Set RUN_EXECUTE = True after the dry-run looks correct.')
